In [1]:
import sys
print(sys.executable)

C:\Users\zorom\.conda\envs\eba-rag\python.exe


In [2]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
import os
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

print("✅ Import completati")

C:\Users\zorom\.conda\envs\eba-rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Import completati


In [3]:
import ssl
import httpx

# Disabilita verifica SSL (solo per rete aziendale con proxy)
import os

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=os.getenv("GROQ_API_KEY"),
    http_client=httpx.Client(verify=False)
)

response = llm.invoke("Dimmi in una frase cos'è una EBA Guideline.")
print(response.content)

Una EBA Guideline (Linea Guida dell'Autorità Bancaria Europea) è un documento che fornisce indicazioni e raccomandazioni alle istituzioni finanziarie e alle autorità di vigilanza dell'Unione Europea su come attuare e interpretare le norme e le regolamentazioni bancarie e finanziarie.


# 1. Let's import the PDF

In [4]:
# Load the pdf
docpfd = PyPDFLoader(
    r"C:/Users/zorom/OneDrive - PROMETEIA SPA/Desktop/AI Course/Consultation paper Guidelines CCF.pdf")

# Check the number of pages
pages = docpfd.load()

In [5]:
# Get all the pages with a table to understand how good is the import
# Trova una pagina che sai contenere una tabella e stampala
page_with_tables = []
for i, page in enumerate(pages):
    if "table" in page.page_content.lower() or "%" in page.page_content:
       # print(f"--- PAGINA {i} ---")
      #  print(page.page_content[:1000])
      #  print()
        page_with_tables.append(i)

# Adjust the medatada. Diventa molto utile quando i documento sono molti bisogna filtrare per anno o tipo
# di parametro o tipo di documento
for page in pages:
    page.metadata["guideline"] = "The EBA consults on Draft Guidelines on the methodology to estimate and apply credit conversion factors under the Capital Requirements Regulation"
    page.metadata["year"] = "2025"
    page.metadata["type"] = "Guidelines"
    page.metadata["topic"] = "Credit Conversion Factor (CCF)"


#Pulisco i metadati
# Campi da tenere
keep_keys = {"source", "page", "page_label", "total_pages", 
             "guideline", "year", "type", "topic"}

for page in pages:
    page.metadata = {k: v for k, v in page.metadata.items() if k in keep_keys}

# Verifica
print(pages[0].metadata)

{'source': 'C:/Users/zorom/OneDrive - PROMETEIA SPA/Desktop/AI Course/Consultation paper Guidelines CCF.pdf', 'total_pages': 114, 'page': 0, 'page_label': '1', 'guideline': 'The EBA consults on Draft Guidelines on the methodology to estimate and apply credit conversion factors under the Capital Requirements Regulation', 'year': '2025', 'type': 'Guidelines', 'topic': 'Credit Conversion Factor (CCF)'}


In [6]:
#----------------------------------------------------------------------------------------------------------
# Per la regulation CCF facciamo un passaggio ulteriore: mettiamo fra i meta dati il capitolo
# introduttivo di background e rationale separato da quello che è la draft regulation
#----------------------------------------------------------------------------------------------------------

In [7]:
# Identifica la pagina di inizio della regulation
for i, page in enumerate(pages):
    if "contents" in page.page_content.lower() or "table of" in page.page_content.lower():
        print(f"Pagina indice {i} (label: {page.metadata['page_label']})")
        print(page.page_content[:1500])
        print()

Pagina indice 1 (label: 2)
CONSULTATION PAPER ON CCF GUIDELINES  
 
 2 
Contents 
1. Responding to this consultation 4 
2. Executive Summary 5 
3. Background and rationale 7 
3.1 Background and introduction 7 
3.2 Interaction with other regulatory products 7 
3.3 Structure of the guidelines 9 
3.4 Chapter 4: Framework for CCF estimation and application 10 
3.5 Chapter 5: Data requirements 16 
3.6 Chapter 6: Risk differentiation 41 
3.7 Chapter 7: Risk quantification 42 
3.8 Chapter 8: CCF for defaulted exposures 48 
3.9 Chapter 9: Treatment of deficiencies and margin of conservatism 49 
3.10 Chapter 10: Downturn CCF estimates 50 
3.11 Chapter 11 & 12: Application of risk parameters 52 
4. Draft guidelines 53 
1. Compliance and reporting obligations 55 
1.1 Status of these guidelines 55 
1.2 Reporting requirements 55 
2. Subject matter, scope and definitions 56 
2.1 Subject matter 56 
2.2 Scope of application 56 
2.3 Addressees 56 
2.4 Definitions 56 
3. Implementation 58 
3.1 Date of a

In [8]:
REGULATION_START = 53  # sostituisci con il numero corretto dopo il controllo sopra

for page in pages:
    page_num = page.metadata["page"]
    
    if page_num < REGULATION_START:
        page.metadata["section"] = "introduction"
    else:
        page.metadata["section"] = "regulation"

# Verifica
print(pages[0].metadata["section"])
print(pages[REGULATION_START].metadata["section"])

introduction
regulation


In [9]:
# Verifica
print(pages[53].metadata)

{'source': 'C:/Users/zorom/OneDrive - PROMETEIA SPA/Desktop/AI Course/Consultation paper Guidelines CCF.pdf', 'total_pages': 114, 'page': 53, 'page_label': '54', 'guideline': 'The EBA consults on Draft Guidelines on the methodology to estimate and apply credit conversion factors under the Capital Requirements Regulation', 'year': '2025', 'type': 'Guidelines', 'topic': 'Credit Conversion Factor (CCF)', 'section': 'regulation'}


In [10]:
pages[53].page_content

'CONSULTATION PAPER ON CCF GUIDELINES \n \n \n 54 \n \n \n \nEBA/GL-REC/20XX/XX \nDD Month YYYY \n \n \nDraft guidelines  \nOn Credit Conversion Factor estimation \nunder Article 182(5) of Regulation (EU) \nNo 575/2013'

# 2. Let's split the text

In [11]:
# Utilizziamo il recursive test splitting per creare i chunck. Per la grandezza del chunck, bisoegnerebbe
# scegliere considerando
# 1) considerare il documento under analysis e le domande che verranno fatti (articolo su articolo o piu 
#    complessi. Primo caso chunkc piccoli secondo piu lunghi ca 1500/2000
# 2) chunk_overlap: di solito 10/20% della lunghezza del chuck
# 3) Considerare il modello di Embedding (esempio HuggingFace  funziona bene fino a 2k di caratteri, poi
#    va peggio 

In [12]:
separatorlist= [
    "\n\n",   # paragrafi
    "\n",     # linee
    ".",      # frasi
    " ",      # parole 
    ""        # fallback
]

chunk_l = 1500
chunk_o = 225

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size    = chunk_l,
    chunk_overlap = chunk_o,
    separators    = separatorlist
)

splits = text_splitter.split_documents(pages)



In [13]:
print(f"Numero di chunk: {len(splits)}")
print(f"\n--- CHUNK 0 ---")
print(splits[20].page_content)
print(f"\nMetadata: {splits[0].metadata}")
print(f"\n--- CHUNK 1 ---")
print(splits[21].page_content)

Numero di chunk: 246

--- CHUNK 0 ---
of the IRB-CCF. 
15. The definition of the CCF-parameter in Article 4(1)(56) provides further guidance pertaining to the 
scope of the CCF estimate, clarifying that the commitment is determined by its advised limit, unless 
the unadvised limit is higher. As such, in order to adhere to the definition of CCF estimates in Article 
4(1)(56) of CRR, in the case where the unadvised limit is higher than the advised limit of which the 
obligor has been informed by the institution, the institution should use the unadvised limit to de-
termine the extent of the commitment, regardless of the unadvised limit not being part of the con-
tractual arrangement. The absence of formal acceptance of the unadvised limit by the client should 
not prevent the institution from modelling such unadvised limit and better refine the scope of i ts 
IRB-CCF.  
16. An unadvised limit is defined as a limit that comprises any credit limit determined by the institution 
and about w

In [14]:
#--------------------------------------------------
# Check if there are chunck without the metadata
#---------------------------------------------------
missing = [i for i, s in enumerate(splits) 
           if "guideline" not in s.metadata or "year" not in s.metadata]
print(f"Chunk senza metadati custom: {len(missing)}")

Chunk senza metadati custom: 0


In [15]:
#------------------------------------------------------------------------------------------------------------
# Mi faccio un'idea di relazione fra pagina (l'indice) e la page_label, quella scritta nel pdf. Altra cosa
# interessante è guardare 
#----------------------------------------------------------------------------------------------------------
for i, chunk in enumerate(splits[:10]):
    print(f"Chunk {i:3d} → page={chunk.metadata['page']}  "
          f"page_label={chunk.metadata['page_label']}  "
          f"chars={len(chunk.page_content)}")

Chunk   0 → page=0  page_label=1  chars=165
Chunk   1 → page=1  page_label=2  chars=1316
Chunk   2 → page=2  page_label=3  chars=1403
Chunk   3 → page=3  page_label=4  chars=1418
Chunk   4 → page=3  page_label=4  chars=315
Chunk   5 → page=4  page_label=5  chars=1430
Chunk   6 → page=4  page_label=5  chars=1435
Chunk   7 → page=4  page_label=5  chars=838
Chunk   8 → page=5  page_label=6  chars=374
Chunk   9 → page=6  page_label=7  chars=1474


In [16]:
#------------------------------------------------------------------------------------------------------------
# I chunck vengono fatti per pagina quindi è per quello che a volte li abbiamo molto corti perchè 
# nella pagina stessa non era stato possibile trovare altri paragrafi
#-----------------------------------------------------------------------------------------------------------
chunk_corti = sorted(splits, key=lambda s: len(s.page_content))

# Stampa i 10 più corti
for i, chunk in enumerate(chunk_corti[:2]):
    print(f"[{i+1}] {len(chunk.page_content):4d} chars | pag.{chunk.metadata['page_label']:>3} | '{chunk.page_content.strip()[:80]}'")

[1]   66 chars | pag. 53 | 'CONSULTATION PAPER ON CCF GUIDELINES 
 
 
 53 
4. Draft guidelines'
[2]   77 chars | pag.  9 | 'CONSULTATION PAPER ON CCF GUIDELINES 
 
 
 9 
3.3 Structure of the guidelines'


In [17]:
#------------------------------------------------------------------------------------------------------------
# Eliminare potenziali chunck che fanno rumore perchè hanno una dimensione molto ridotta. Facciamo alcune
# prove sulla dimensione.
#-------------------------------------------------------------------------------------------------------------
MIN_CHARS = 210

splits_clean = [s for s in splits if len(s.page_content.strip()) >= MIN_CHARS]

print(f"Chunk originali:  {len(splits)}")
print(f"Chunk rimossi:    {len(splits) - len(splits_clean)}")
print(f"Chunk finali:     {len(splits_clean)}")

Chunk originali:  246
Chunk rimossi:    4
Chunk finali:     242


In [18]:
#----------------------------------------------------------------------------------------------------------
# Fatte alcune stampe per cercare di capire quali sono i chunck che vengono eliminati e se potevano
# avere importanza
#-----------------------------------------------------------------------------------------------------------
print("\nChunk eliminati:")
for s in splits:
    if len(s.page_content.strip()) < MIN_CHARS:
        print(f"  pag.{s.metadata['page_label']:>3} | {len(s.page_content):3d} chars | '{s.page_content.strip()}'")


Chunk eliminati:
  pag.  1 | 165 chars | 'EBA/CP/2025/10 
02 July 2025 
 
 
Consultation paper  
Draft guidelines 
on Credit Conversion Factor estimation under Article 182(5) of Reg-
ulation (EU) No 575/2013'
  pag.  9 |  77 chars | 'CONSULTATION PAPER ON CCF GUIDELINES 
 
 
 9 
3.3 Structure of the guidelines'
  pag. 53 |  66 chars | 'CONSULTATION PAPER ON CCF GUIDELINES 
 
 
 53 
4. Draft guidelines'
  pag. 54 | 203 chars | 'CONSULTATION PAPER ON CCF GUIDELINES 
 
 
 54 
 
 
 
EBA/GL-REC/20XX/XX 
DD Month YYYY 
 
 
Draft guidelines  
On Credit Conversion Factor estimation 
under Article 182(5) of Regulation (EU) 
No 575/2013'


# 3. Embedding e Vector Store

In [19]:
# Richiama l'API OpenAI e le directory 
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
CHROMA_DIR = r"C:/Users/zorom/OneDrive - PROMETEIA SPA/Desktop/AI Course/chroma_db"
os.makedirs(CHROMA_DIR, exist_ok=True)

print("✅ Setup completato")
print(f"   Chroma directory: {CHROMA_DIR}")

✅ Setup completato
   Chroma directory: C:/Users/zorom/OneDrive - PROMETEIA SPA/Desktop/AI Course/chroma_db


In [20]:
#------------------------------------------------------------------------------------------------------------
# Scelta del modello di embedding
#------------------------------------------------------------------------------------------------------------
# Crea il modello di embedding
embeddings = OpenAIEmbeddings(
     model="text-embedding-3-large"
    ,http_client=httpx.Client(verify=False)
)

# Verifica. Open AI Large trasforma ogni "chuck, parola" qualsiasi testo passatogli in input in un 
# vettore sempre di dimensione 3072. I tooken limite 8191.  circa 6000 parole.
test = embeddings.embed_query("test")
print(str(len(test)))
#------------------------------------------------------------------------------------------------------------

3072


In [21]:
#----------------------------------------------------------------------------------------------------------
#  Salva il vector store: Nota questo codice va girato solo per il primo embedding. Successivamente
#  girare la cella sotto. Ovviamente se cambiano i chunck, il documento o qualsiasi cosa va rigirata
#  questa cella
#-----------------------------------------------------------------------------------------------------------
# Occhio a potenziali db gia salvati
vectorstore = Chroma.from_documents(
    documents         = splits_clean,
    embedding         = embeddings,
    persist_directory = CHROMA_DIR
)

# Verifica se l'embedding a le numerosità attese
print(f"Documenti indicizzati: {vectorstore._collection.count()}")
print(f"Chunk totali: {len(splits)}")
#-----------------------------------------------------------------------------------------------------------

Documenti indicizzati: 484
Chunk totali: 246


In [22]:
#----------------------------------------------------------------------------------------------------------
# Retriver test: faccio delle domande e cerco di capire quali sono i chunck che vengono selezionati
#-----------------------------------------------------------------------------------------------------------
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
#docs = retriever.invoke("What is the credit conversion factor?")
#docs = retriever.invoke("Which is the definition of the anadvise limit?")
docs = retriever.invoke("How should be treated the so called 'fast defaults'?")

for i, doc in enumerate(docs):
    print(f"\n--- Risultato {i+1} | pag.{doc.metadata['page_label']} ---")
    print(doc.page_content[:1500])
    print()
#-----------------------------------------------------------------------------------------------------------


--- Risultato 1 | pag.30 ---
CONSULTATION PAPER ON CCF GUIDELINES 
 
 
 30 
Consultation box 
A multiple default treatment is also aligned with the concept of a fixed reference date for a facility 
defined as 12 months prior to the date of default. Not introducing a multiple default treatment 
would either imply the exclusion of all those defaults on a facility occurring within 12 months after 
the exit date of a preceding default on that facility, or it would imply an analysis of the retraction of 
the reference date of those defaults. As such, it could also be argued to extend the dependence 
period to identify defaults on a facility that should be merged from 9 to 12 months. However, a 
consistent approach with the LGD multiple default treatment was chosen, because the number of 
facilities defaulting between 9 to 12 months after the exit date from their first default is likely to be 
small.  
An issue with the multiple default treatment is that the drawing behaviour in between the

In [23]:
#----------------------------------------------------------------------------------------------------------
# Potrebbe sembrare controintuitvo che la definizione di unadvise limit data a pagina 56 non è la prima
# ma nel primo chunck la parola unadvise limit viene ripetuta piu volte aumentando pertanto la similarità
# Lanciando il pezzo sotto di codice si puo vedere come in realtà i vector store sono tutti molto simili
#
#docs_with_scores = vectorstore.similarity_search_with_score(
#    "Which is the definition of the unadvised limit?", 
#    k=5
#)
#
#for i, (doc, score) in enumerate(docs_with_scores):
#    print(f"\n--- Risultato {i+1} | score={score:.4f} | pag.{doc.metadata['page_label']} ---")
#    print(doc.page_content[:200])
#
#---------------------------------------------------------------------------------------------------------

In [24]:
#------------------------------------------------------------------------------------------------------------
# Il retriver MMR sembra dare risultati migliori rispetto il retriver standard da similarity. Se la domanda
# è semplice i risultati sono molto comparabili ma quando facciamo domande piu complesse qualche vantaggio
# si vede e sono recuperati chunck un po' diversi me piu "corretti". Proseguiamo quindi con MMR
#-----------------------------------------------------------------------------------------------------------
retriever = vectorstore.as_retriever(
    search_type   = "mmr",
    search_kwargs = {
        "k"       : 5,
        "fetch_k" : 20
    }
)

# Test
#docs = retriever.invoke("Which is the definition of the unadvised limit?")
docs = retriever.invoke("How should be treated the so called 'fast defaults'?")

for i, doc in enumerate(docs):
    print(f"\n--- Risultato {i+1} | pag.{doc.metadata['page_label']} ---")
    print(doc.page_content[:1500])


--- Risultato 1 | pag.30 ---
CONSULTATION PAPER ON CCF GUIDELINES 
 
 
 30 
Consultation box 
A multiple default treatment is also aligned with the concept of a fixed reference date for a facility 
defined as 12 months prior to the date of default. Not introducing a multiple default treatment 
would either imply the exclusion of all those defaults on a facility occurring within 12 months after 
the exit date of a preceding default on that facility, or it would imply an analysis of the retraction of 
the reference date of those defaults. As such, it could also be argued to extend the dependence 
period to identify defaults on a facility that should be merged from 9 to 12 months. However, a 
consistent approach with the LGD multiple default treatment was chosen, because the number of 
facilities defaulting between 9 to 12 months after the exit date from their first default is likely to be 
small.  
An issue with the multiple default treatment is that the drawing behaviour in between the

# 4. Q&A Session 

In [25]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [26]:
#-------------------------------------------------------------------------------------------------------
# Definiamo il prompt. E' molto importante definirlo bene e soprattutto specificare l'output che si
# vuole ricevere. Importante anche specificare 
#--------------------------------------------------------------------------------------------------------
prompt_template = """You are an expert assistant specialized in EBA Guidelines and banking regulation.
Answer the question using ONLY the context provided below. Respond in the same language as the question.
Always cite the page number and article number when available at the end of the answer within brackets.
If you cannot find the answer in the context, respond exactly with: "I'm not able to find the answer"
Context: 
{context}
Question: 
{question}
Answer:"""

#Creo il prompt template passando in input (opvvero quelli dentro parentesi graffe)
prompt = PromptTemplate(
    template       = prompt_template,
    input_variables= ["context", "question"]
)

In [27]:
#--------------------------------------------------------------------------------------------------------
# Creiamo la rag chain
#-------------------------------------------------------------------------------------------------------
def format_docs(docs):
    return "\n\n".join([
        f"[Page {doc.metadata['page_label']}]\n{doc.page_content}" 
        for doc in docs
    ])

# Sta costruendo una chain con LCEL. Il simbolo | è il modo di LangChain per collegare i componenti in 
# sequenza — funziona come una pipe, l'output di un componente diventa l'input del successivo.
# retriever | format_docs    → recupera i chunk e li formatta con [Page X]
# RunnablePassthrough()      → passa la domanda invariata
# prompt                     → assembla contesto + domanda nel template
# llm                        → genera la risposta
# StrOutputParser()          → estrae il testo dalla risposta dell'LLM

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
#---------------------------------------------------------------------------------------------------------
#---------------------------------------------------------------------------------------------------------

In [28]:
response = rag_chain.invoke("Which is the definition of the unadvised limit?")
print(response)

An unadvised limit is defined as a limit that comprises any credit limit determined by the institution and about which the obligor has not been informed by the institution and according to which additional drawings beyond the advised limit are at least temporarily possible [Page 12, Article 16].


In [29]:
response = rag_chain.invoke("Can you tell me what regulation says about representativeness?")
print(response)

The regulation emphasizes the importance of representativeness in the data used for the estimation of risk parameters. Institutions should have sound policies, processes, and methods for assessing the representativeness of data used for this purpose. The representativeness requirements apply to the data used to test the model performance as a whole, and institutions should either make adjustments to the data or select a different testing sample if a lack of representativeness is observed. The regulation also states that the representativeness requirements are simplified for CCF in comparison to the GL PD and LGD, and are intended to mitigate model risks related to non-representative data [Page 18, Page 63, Article 174(b) of Regulation (EU) No 575/2013, Articles 174(c), 179(1)(d), 179(2)(b) and 182(1)(h) of Regulation (EU) No 575/2013].


In [30]:
response = rag_chain.invoke("Can you elaborate more on that?")
print(response)

I'm not able to find the answer


In [31]:
response = rag_chain.invoke("Which is the definition of the unadvised limit? Please use all the possible referencies you can find")
print(response)

The definition of the unadvised limit is a limit that comprises all of the following criteria: 
• any credit limit determined by the institution and  
• about which the obligor has not been informed by the institution and  
• according to which additional drawings beyond the advised limit are at least temporarily possible. 
This definition is mentioned in page 12, and also in page 56, article 2.4, and page 74, article 4(1)(56) of Regulation (EU) No 575/2013. 
Additionally, article 16 on page 12, and article 72 on page 74 provide further guidance on the definition and application of the unadvised limit. 
It is also important to note that internally defined limits related to the risk appetite of the institution where additional drawings by the obligor on its revolving line are not possible beyond the advised limit should not be considered as an unadvised limit, as stated in article 17 on page 12 [Page 12, article 16, Page 56, article 2.4, Page 74, article 4(1)(56), Page 12, article 17].


In [ ]:
response = rag_chain.invoke("Quale è la definizione di unadvise limit?")
print(response)

# 5. Q&A con Memoria
Utilizziamo `ConversationSummaryBufferMemory` per mantenere il contesto della conversazione.
- Mantiene per intero le ultime interazioni fino a `max_token_limit` token
- Riassume le interazioni più vecchie quando il limite viene superato
- `ConversationalRetrievalChain` riformula automaticamente le domande di follow-up tenendo conto della storia

In [ ]:
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationSummaryBufferMemory

#----------------------------------------------------------------------------------------------------------
# ConversationSummaryBufferMemory — ibrido tra buffer e summary:
# - mantiene per intero le ultime interazioni fino a max_token_limit token
# - riassume quelle più vecchie quando il limite viene superato
# Parametri:
#   llm             → modello usato per generare il riassunto (Groq, gratuito)
#   max_token_limit → token massimi prima di iniziare a riassumere (~3-4 Q&A)
#   memory_key      → nome con cui la storia viene passata alla chain
#   return_messages → salva la storia come lista di messaggi strutturati
#   output_key      → quale campo della risposta salvare (answer vs source_documents)
#----------------------------------------------------------------------------------------------------------
memory = ConversationSummaryBufferMemory(
    llm             = llm,
    max_token_limit = 1000,
    memory_key      = "chat_history",
    return_messages = True,
    output_key      = "answer"
)

print("✅ Memoria inizializzata")

In [ ]:
#----------------------------------------------------------------------------------------------------------
# ConversationalRetrievalChain — estende RetrievalQA aggiungendo la memoria.
# Prima di interrogare il vector store, riformula la domanda tenendo conto della storia:
#   'Can you elaborate more on that?' 
#   → 'Can you elaborate more on the unadvised limit?'
# return_source_documents=True → restituisce anche i chunk usati per generare la risposta
#----------------------------------------------------------------------------------------------------------
rag_chain_memory = ConversationalRetrievalChain.from_llm(
    llm                     = llm,
    retriever               = retriever,
    memory                  = memory,
    return_source_documents = True,
    verbose                 = False
)

print("✅ Chain con memoria pronta")

In [ ]:
# Prima domanda
response = rag_chain_memory.invoke({"question": "Which is the definition of the unadvised limit?"})
print(response["answer"])

In [ ]:
# Follow-up senza contesto esplicito — senza memoria rispondeva 'I'm not able to find the answer'
# Con memoria riformula la domanda tenendo conto della storia
response = rag_chain_memory.invoke({"question": "Can you elaborate more on that?"})
print(response["answer"])

In [ ]:
# Verifica fonti usate per generare la risposta
print("Fonti utilizzate:")
for doc in response["source_documents"]:
    print(f"  pag.{doc.metadata['page_label']} | {doc.page_content[:100]}...")

In [ ]:
# Per resettare la memoria e iniziare una nuova conversazione
memory.clear()
print("✅ Memoria resettata")